In [19]:
import subprocess
import sys
import random as rd
import webbrowser

def instalar(package): #Instala o pacote necessário caso ele não esteja instalado
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import pandas as pd
except ImportError:
    instalar('pandas')
    import pandas as pd

In [20]:
#Cria um backup para garantir que a coleção não seja sobreescrita acidentalmente
try: 
    backup = pd.read_csv('save.csv')
except:
    backup = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt']) 
backup.to_csv('save_backup.csv', index=False)

In [21]:
#CSV que armazena o nome dos sets e seus links
direct = pd.read_csv('packs\direct.csv')

In [22]:
def escolhe_pacote(name):
    view = input('Deseja ver as cartas desse pacote na Wiki?(s/n): ')
    if view == 's':
        link = direct[direct['set'] == name.upper()]['link'].values[0]
        webbrowser.open(link)
    else:
        data = pd.read_csv(f'packs\{name}.csv')
        for index, row in data.iterrows(): 
            print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]}')


In [23]:
def gerar_link_wiki(nome_carta):
    # A wiki usa underscores no lugar de espaços
    nome_formatado = nome_carta.replace(' ', '_')
    return f'https://cardfight.fandom.com/wiki/{nome_formatado}'

In [24]:
def ver_colecao(nome, web=False):
    try:
        save = pd.read_csv(f'{nome}.csv')
        if web == False:
            for index, row in save.iterrows():
                print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]} - Qtt: {row["qtt"]}')
        else:
            print(f'Abrindo a Wiki de: {len(save)} cartas...')
            for index, row in save.iterrows():
                link = gerar_link_wiki(row['name'])
                webbrowser.open(link) 
    except:
        print('Você ainda não tem nenhuma carta na coleção! Abra alguns pacotes para começar a colecionar!')

In [25]:
""" def web_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            link = gerar_link_wiki(row['name'])
            print(f'Abrindo Wiki de: {row["name"]}...')
            webbrowser.open(link) # Isso abre o navegador automaticamente
    except:
        print('Erro ao carregar coleção.') """

' def web_colecao(nome):\n    try:\n        save = pd.read_csv(f\'{nome}.csv\')\n        for index, row in save.iterrows():\n            link = gerar_link_wiki(row[\'name\'])\n            print(f\'Abrindo Wiki de: {row["name"]}...\')\n            webbrowser.open(link) # Isso abre o navegador automaticamente\n    except:\n        print(\'Erro ao carregar coleção.\') '

In [26]:
def filtra_colecao(tipo, filtro, data = 'save'):
    if data == 'save':
        save = pd.read_csv(f'{data}.csv')
    else:
        save = data.copy()
    
    if (tipo == 'type') and (filtro.lower() == 'trigger'):
        save = save[(save['type'].str.lower()) != 'normal unit']
    elif (tipo == 'grade'):
        save = save[(save['grade'] == int(filtro))]
    else:
        save = save[(save[tipo].str.lower()) == filtro]

    if save.empty:
        print(f'Nenhuma carta encontrada para o filtro: {tipo} = {filtro}')
    else:
        for index, row in save.iterrows():
            print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]} - Qtt: {row["qtt"]}')
    return save    

In [27]:
def atualizar_save(box, pacote):
    #Quero colocar os valores do pacote na box e se houver repetidos, qtt += 1
    
    for index, row in pacote.iterrows():
        card = box[(box['set'] == row['set']) & (box['id'] == row['id'])]
        if not card.empty:
            box.loc[card.index, 'qtt'] += 1
        else:
            new_card = row.to_dict()
            new_card['qtt'] = 1
            #box não tem append, então vou criar um novo dataframe com a nova linha e concatenar com a box
            new_card_df = pd.DataFrame([new_card])
            box = pd.concat([box, new_card_df], ignore_index=True)

    return box

In [28]:
def rodar_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    pacote = []
    

    commons = data[data['rarity'] == 'C']
    c_tiradas = commons.sample(n=4).to_dict('records')
    pacote.extend(c_tiradas)

    luck = rd.randint(1, 100)

    if luck <= 70:
        raridade = 'R'
    elif luck <= 90:
        raridade = 'RR'
    elif luck <= 99:
        raridade = 'RRR'
    else:
        raridade = 'SP'

    raras = data[data['rarity'] == raridade]
    r_tirada = raras.sample(n=1).to_dict('records')
    pacote.extend(r_tirada)

    pacote = pd.DataFrame(pacote)
    
    return pacote

In [29]:
def rodar_box(name, qtt):
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
    
    for i in range(qtt):
        pacote = rodar_pacote(name)
        box = atualizar_save(box, pacote)
    
    try:
        save = pd.read_csv('save.csv')
    except:
        save = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])

    box.sort_values(by=['id'], inplace=True)
    box.to_csv('last_box.csv', index=False)

    save = atualizar_save(save, box)
    save.sort_values(by=['set', 'id'], inplace=True)
    save.to_csv('save.csv', index=False)
    
    return box

In [2]:
pip install duckduckgo_search

     -------------------------------------- 108.3/108.3 kB 2.1 MB/s eta 0:00:00
     ---------------------------------------- 3.9/3.9 MB 2.6 MB/s eta 0:00:00
     ---------------------------------------- 4.0/4.0 MB 3.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from duckduckgo_search import DDGS
from PIL import Image
import requests
from io import BytesIO

def busca_rapida_ddg(termo):
    with DDGS() as ddgs:
        # Busca imagens no DuckDuckGo (que indexa resultados similares ao Google)
        resultados = list(ddgs.images(termo, max_results=1))
        if resultados:
            url = resultados[0]['image']
            img = Image.open(BytesIO(requests.get(url).content))
            img.show()

busca_rapida_ddg("Blaster Blade")

C:\Users\Cauã Gomes\AppData\Local\Temp\ipykernel_7372\67869676.py:7: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


RatelimitException: https://duckduckgo.com/i.js?o=json&q=Blaster+Blade&l=us-en&vqd=4-61388331830776574341446320957115758707&p=1&f=%2C%2C%2C%2C%2C 403 Ratelimit

In [12]:
import tkinter as tk
import requests
from io import BytesIO
from PIL import Image, ImageTk

def exibir_carta_da_url(url_da_imagem):
    # 1. Cria a janela ou frame principal
    janela = tk.Tk()
    janela.title("Simulador CFV")

    try:
        # 2. Faz o download da imagem apenas para a memória (RAM)
        resposta = requests.get(url_da_imagem)
        resposta.raise_for_status() 

        # 3. Converte os bytes em uma imagem usando o Pillow
        dados_imagem = BytesIO(resposta.content)
        imagem_pil = Image.open(dados_imagem)

        # Opcional: Redimensionar a carta para manter o padrão no simulador
        # imagem_pil = imagem_pil.resize((250, 350), Image.Resampling.LANCZOS)

        # 4. Converte para o formato de exibição do Tkinter
        imagem_tk = ImageTk.PhotoImage(imagem_pil)

        # 5. Exibe a imagem em um Label
        label_imagem = tk.Label(janela, image=imagem_tk)
        
        # IMPORTANTE: Manter a referência da imagem para o Garbage Collector não apagá-la
        label_imagem.image = imagem_tk 
        label_imagem.pack(padx=10, pady=10)

    except Exception as e:
        print(f"Erro ao carregar a imagem da carta: {e}")
        label_erro = tk.Label(janela, text="Erro ao carregar imagem")
        label_erro.pack()

    janela.mainloop()

# Substitua pela URL real de uma carta armazenada em algum banco de dados ou Wiki
url_teste = "https://static.wikia.nocookie.net/cardfight/images/6/67/D-BT01-001-EN-RRR.png"
exibir_carta_da_url(url_teste)

Erro ao carregar a imagem da carta: 404 Client Error: Not Found for url: https://static.wikia.nocookie.net/cardfight/images/6/67/D-BT01-001-EN-RRR.png
